# EMBASE Collection Pipeline

*Author: Regina Chua*

> This notebook is the EMBASE arm of the systematic review and uses the **Embase Search API**
> (`api.elsevier.com/content/embase/article`) to pull records programmatically. It mirrors the
> structure of `pubmed.ipynb` and `scopus.ipynb`: all search criteria are imported from
> `search_strategy.py`, the query is translated into Embase CommandLanguage (the same syntax used
> on Embase.com), and results are cleaned and exported to a CSV that slots into the deduplication
> step (Milestone 4).

**API access:** you need your **Elsevier API key** and, if outside the institution's network,
an **insttoken**. Both live in `.env` (see Section 1). The same API key you provided to
`pybliometrics` for SCOPUS works here too.

**Pagination:** the API returns up to 200 records per request; the collector loops through
pages automatically.

**References:** [Elsevier Embase API](https://dev.elsevier.com/embase_apis.html).

## 1. Environment Setup

> The Embase API uses the same Elsevier API key as SCOPUS. I read it from `.env` as
> `ELSEVIER_API_KEY` (and optionally `ELSEVIER_INSTTOKEN` for off-campus access). If you've
> already configured `pybliometrics`, the key you gave it is the right one — just add it to `.env`.
> `EMBASE_READY` guards the collection cell so the notebook runs cleanly even before the key is set.

In [ ]:
import os
import time
from datetime import datetime
from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv

from search_strategy import (
    INCLUSION_CRITERIA,
    ALTERNATE_TERMS,
    EXCLUSION_TERMS,
    DATE_FILTER,
    CLEANING_RULES,
)

load_dotenv()
pd.set_option("display.max_colwidth", 120)

ELSEVIER_API_KEY = os.getenv("ELSEVIER_API_KEY")
ELSEVIER_INSTTOKEN = os.getenv("ELSEVIER_INSTTOKEN")  # optional — needed off-campus
EMBASE_READY = bool(ELSEVIER_API_KEY)

print("Environment ready.")
print("ELSEVIER_API_KEY set:", EMBASE_READY)
print("ELSEVIER_INSTTOKEN set:", bool(ELSEVIER_INSTTOKEN))
if not EMBASE_READY:
    print(
        "\nAdd the following to your .env file and re-run this cell:\n"
        "  ELSEVIER_API_KEY=your_key_here\n"
        "  ELSEVIER_INSTTOKEN=your_insttoken_here  # if off-campus"
    )

## 2. Build Query

> The Embase API uses the same **CommandLanguage** as Embase.com, so the `:ti,ab` field tag
> (title + abstract) carries over directly. I build the same three-group
> `(disease) AND (spatial) AND (exposure) NOT (exclusions)` structure as the other notebooks, with
> a `[YEAR]/py` date filter appended. The wildcard `*` is supported, so terms like `parkinson*`
> are passed through unchanged.

In [ ]:
def merge_terms(primary, alternates):
    """Combine primary + alternate/NLP terms per category (order-preserving,
    case-insensitive de-duplication). Mirrors pubmed.ipynb."""
    merged = {}
    for category, base_terms in primary.items():
        seen, combined = set(), []
        for t in list(base_terms) + list(alternates.get(category, [])):
            if t.lower() not in seen:
                seen.add(t.lower())
                combined.append(t)
        merged[category] = combined
    return merged


def embase_group(terms):
    """OR-group of terms with the Embase ':ti,ab' (title/abstract) field tag."""
    return "(" + " OR ".join(f"'{t}':ti,ab" for t in terms) + ")"


def build_embase_query(inclusion, exclusion, date_filter):
    """Compose the Embase CommandLanguage query.

    Structure: (disease):ti,ab AND (spatial):ti,ab AND (exposure):ti,ab
               NOT (exclusions):ti,ab AND [start-end]/py
    """
    include = " AND ".join([
        embase_group(inclusion["disease"]),
        embase_group(inclusion["spatial"]),
        embase_group(inclusion["exposure"]),
    ])
    q = f"{include} NOT {embase_group(exclusion)}"
    start_year = int(date_filter["start_date"][:4])
    end_year = int(date_filter["end_date"][:4]) if date_filter.get("end_date") else datetime.now().year
    q += f" AND [{start_year}-{end_year}]/py"
    return q


INCLUDE_ALTERNATE_TERMS = True
search_criteria = (
    merge_terms(INCLUSION_CRITERIA, ALTERNATE_TERMS)
    if INCLUDE_ALTERNATE_TERMS
    else INCLUSION_CRITERIA
)

query = build_embase_query(search_criteria, EXCLUSION_TERMS, DATE_FILTER)

print("Alternate terms folded into query:", INCLUDE_ALTERNATE_TERMS)
for category, term_list in search_criteria.items():
    print(f"{category} terms ({len(term_list)}):", term_list)
print(f"\nQuery ({len(query)} chars):\n", query)

## 3. Collect Articles from EMBASE

> The Embase API paginates with `start` (offset) and `count` (page size, max 200). I use POST
> rather than GET because our query string is long. The `apiKey` and optional `insttoken` go in the
> request headers. I pause one second between pages to stay within the API's sequential-only
> request requirement (parallel requests are disallowed by Elsevier).
>
> A `RESPONSE_INSPECT` flag lets you print the raw first-page JSON so you can verify the field
> names returned by your subscription — the flattener below covers the common patterns but field
> availability varies by access level.

In [ ]:
EMBASE_API_URL = "https://api.elsevier.com/content/embase/article"
PAGE_SIZE = 200       # API maximum
MAX_RECORDS = 5000    # safety cap
SLEEP_BETWEEN_PAGES = 1.0
RESPONSE_INSPECT = False  # set True to print the raw first-page response for debugging


def _get(obj, *keys, default=None):
    """Safe nested get that handles both dict keys and list-valued fields."""
    for k in keys:
        if not isinstance(obj, dict):
            return default
        obj = obj.get(k, default)
        if obj is None:
            return default
    return obj


def _authors_str(raw):
    """Flatten the authors field to a semicolon-separated string."""
    if isinstance(raw, list):
        parts = []
        for a in raw:
            if isinstance(a, dict):
                parts.append(a.get("author-name") or a.get("$") or str(a))
            else:
                parts.append(str(a))
        return "; ".join(parts)
    return raw


def flatten_embase(rec):
    """Flatten one Embase API result record into the shared schema.

    The API may return fields as flat keys or under Dublin Core / PRISM
    namespaces depending on the subscription view — both are handled here.
    """
    # Title
    title = (
        rec.get("dc:title")
        or rec.get("title")
        or _get(rec, "titles", "title")
    )
    # Abstract
    abstract = (
        rec.get("dc:description")
        or rec.get("abstract")
        or _get(rec, "abstracts", "abstract")
    )
    # DOI
    doi = (
        rec.get("prism:doi")
        or rec.get("doi")
        or _get(rec, "identifier", "doi")
    )
    # Publication date
    pub_date = (
        rec.get("prism:coverDate")
        or rec.get("publicationDate")
        or rec.get("prism:coverDisplayDate")
    )
    # Journal
    journal = (
        rec.get("prism:publicationName")
        or rec.get("journalTitle")
        or _get(rec, "source", "sourcetitle")
    )
    # Authors
    authors = _authors_str(
        rec.get("dc:creator")
        or rec.get("authors")
        or _get(rec, "authors", "author")
    )
    # Keywords (author keywords)
    kw_raw = rec.get("authkeywords") or rec.get("keywords")
    keywords = kw_raw if isinstance(kw_raw, str) else (
        "; ".join(str(k) for k in kw_raw) if isinstance(kw_raw, list) else None
    )
    # PubMed ID
    pmid = rec.get("pubmed-id") or rec.get("medline-pmid") or rec.get("pmid")
    # Citations
    cited_by = rec.get("citedby-count") or rec.get("numCitations")

    return {
        "title":            title,
        "abstract":         abstract,
        "publication_date": pub_date,
        "authors":          authors,
        "journal":          journal,
        "doi":              doi,
        "url":              f"https://doi.org/{doi}" if doi else None,
        "num_citations":    cited_by,
        "pubmed_id":        pmid,
        "keywords":         keywords,
        "source":           "embase",
    }


def fetch_embase(query, api_key, insttoken=None, max_records=MAX_RECORDS):
    """Page through the Embase API and return a flat list of result dicts."""
    headers = {
        "X-ELS-APIKey":    api_key,
        "Accept":          "application/json",
        "Content-Type":    "application/x-www-form-urlencoded",
    }
    if insttoken:
        headers["X-ELS-Insttoken"] = insttoken

    records, start = [], 0
    total = None

    while len(records) < max_records:
        data = {"query": query, "start": start, "count": PAGE_SIZE}
        resp = requests.post(EMBASE_API_URL, headers=headers, data=data, timeout=60)
        resp.raise_for_status()
        payload = resp.json()

        if RESPONSE_INSPECT and start == 0:
            import json as _json
            print("--- raw first-page response (first 2000 chars) ---")
            print(_json.dumps(payload, indent=2)[:2000])

        # The Embase API nests results under 'search-results' or directly at root.
        root = payload.get("search-results") or payload

        if total is None:
            total = int(root.get("opensearch:totalResults") or root.get("resultsFound") or 0)
            print(f"Embase reports {total} matching records for this query.")

        entries = root.get("entry") or root.get("results") or []
        if not entries:
            break

        records.extend(flatten_embase(e) for e in entries)
        print(f"  page starting at {start}: +{len(entries)} ({len(records)}/{min(total, max_records)})")

        if len(records) >= total:
            break
        start += PAGE_SIZE
        time.sleep(SLEEP_BETWEEN_PAGES)

    return records


run_ts = datetime.now().isoformat(timespec="seconds")
df_raw = pd.DataFrame()

if not EMBASE_READY:
    print("Skipping collection — ELSEVIER_API_KEY is not set (see Section 1).")
else:
    try:
        print(f"Querying Embase API at {run_ts} ...")
        records = fetch_embase(query, ELSEVIER_API_KEY, insttoken=ELSEVIER_INSTTOKEN)
        df_raw = pd.DataFrame(records)
        print(f"\nDownloaded {len(df_raw)} records total.")
    except requests.HTTPError as exc:
        print(
            f"Embase API returned an error: {exc}\n"
            "Common causes:\n"
            "  401 — API key missing or wrong; check ELSEVIER_API_KEY in .env\n"
            "  403 — institutional access required; add ELSEVIER_INSTTOKEN to .env\n"
            "  429 — weekly quota exceeded; wait and retry"
        )
    except Exception as exc:  # noqa: BLE001
        print(f"Request failed: {type(exc).__name__}: {exc}")

preview_cols = [c for c in ["title", "publication_date", "journal", "doi"] if c in df_raw.columns]
if not df_raw.empty:
    print()
    display(df_raw[preview_cols].head())

## 4. Inspect Raw Response Fields

> If the flattener in Section 3 leaves many columns empty it's likely because your subscription
> returns fields under slightly different names. Run this cell to see exactly what keys came back in
> the first record, then adjust `flatten_embase()` accordingly. Set `RESPONSE_INSPECT = True` in
> Section 3 to also print the full raw JSON of the first page.

In [ ]:
if df_raw.empty:
    print("No data yet — run Section 3 first.")
else:
    print("Columns returned:", list(df_raw.columns))
    print("\nNull counts:")
    print(df_raw.isnull().sum())
    print(f"\nAbstracts populated: {df_raw['abstract'].notna().sum()} / {len(df_raw)}")

## 5. Clean Results

> Same cleaning contract as the other notebooks: drop rows with no title, deduplicate on normalised
> title, and (per `CLEANING_RULES`) require a DOI so every record is uniquely identifiable for the
> cross-database merge in Milestone 4.

In [ ]:
df_clean = df_raw.copy()

if not df_clean.empty:
    df_clean = df_clean.dropna(subset=["title"])
    if CLEANING_RULES.get("remove_duplicate_titles", True):
        df_clean["_title_lower"] = df_clean["title"].astype(str).str.lower().str.strip()
        df_clean = df_clean.drop_duplicates(subset=["_title_lower"]).drop(columns=["_title_lower"])
    if CLEANING_RULES.get("require_doi", True) and "doi" in df_clean.columns:
        df_clean = df_clean.dropna(subset=["doi"])

print(f"Records after cleaning: {len(df_clean)}  (from {len(df_raw)} raw)")
if not df_clean.empty:
    display(df_clean[preview_cols].head())

## 6. Export

> Save the cleaned set to CSV with the shared column layout for the Milestone 4 deduplication merge.
> Only writes when there is data so a failed run doesn't overwrite a previous good export.

In [ ]:
output_path = Path("embase_results_2026.csv")

if df_clean.empty:
    print("Nothing to export — df_clean is empty (see Sections 1 and 3).")
else:
    df_clean.to_csv(output_path, index=False)
    print(f"Exported {len(df_clean)} records to {output_path.resolve()}")
    print(f"Run timestamp: {run_ts}")

## 7. Notes & Next Steps

> - **API key setup:** add `ELSEVIER_API_KEY=your_key` (and `ELSEVIER_INSTTOKEN=your_token` if
>   off-campus) to `.env`. The key is the same one already given to `pybliometrics` for SCOPUS.
>   Manage keys at https://dev.elsevier.com/apikey/manage.
> - **Response fields:** if abstracts or keywords come back empty, flip `RESPONSE_INSPECT = True`
>   in Section 3 and run again to see the raw JSON structure. Adjust the fallback chains in
>   `flatten_embase()` to match the actual keys returned by your subscription.
> - **Quota:** the Embase API has a weekly quota. Requests are sequential only (no parallel calls).
>   `pybliometrics` does not manage this quota — it applies to the raw API separately.
> - **Emtree terms:** the current query uses free-text `:ti,ab` searches. For a more sensitive
>   final search, map the disease/exposure terms to **Emtree** subject headings
>   (e.g. `'parkinson disease'/exp`) and combine them with OR — agree this with Malcolm & Lukas.
> - **Schema:** export columns match `pubmed_results_cleaned_2026.csv`, `scopus_results_2026.csv`,
>   and `web_of_science_results_2026.csv` for clean concatenation in Milestone 4.